# Backtest Statistics

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.backtesting.backtest_statistics import (
    ClassificationScores,
    Efficiency,
    GeneralCharacteristics,
    ImplementationShortfall,
    Performance,
    Runs,
)

from src.backtesting.portfolio import (
    daily_portfolio,
    portfolio_equity,
    portfolio_trades,
)


def registered_metric_order(metric_class, labels):
    return [
        name
        for name, value in vars(metric_class).items()
        if not name.startswith("_")
        and isinstance(value, staticmethod)
        and name in labels
    ]


result_dir = PROJECT_ROOT / "data/backtest_results"
returns = pd.read_parquet(
    result_dir / "event_strategy_returns.parquet"
).sort_index()
holdout = returns[returns["partition"].eq("holdout")].copy()
strategies = ["primary_only", "meta_filtered"]
annual_risk_free_rate = 0.03
annualized_benchmark_sharpe_ratio = 1.0
portfolio_ledger = pd.read_parquet(
    result_dir / "portfolio_ledger.parquet"
)
ledgers = {
    strategy: portfolio_ledger.xs(strategy, level="strategy").sort_index()
    for strategy in strategies
}
daily = {strategy: daily_portfolio(ledger) for strategy, ledger in ledgers.items()}
trades = {strategy: portfolio_trades(ledger) for strategy, ledger in ledgers.items()}
equity = {strategy: portfolio_equity(ledger) for strategy, ledger in ledgers.items()}
evaluation_index = ledgers[strategies[0]].index
start = GeneralCharacteristics.start(evaluation_index)
end = GeneralCharacteristics.end(evaluation_index)
elapsed_years = (end - start).total_seconds() / (365.25 * 24 * 60 * 60)
initial_aum = {
    strategy: ledger["aum_before"].iloc[0]
    for strategy, ledger in ledgers.items()
}
for strategy in strategies:
    if not ledgers[strategy].index.equals(ledgers[strategies[0]].index):
        raise ValueError("Both strategies must use the same observed price grid.")


## General Characteristics

- **Purpose:** Describe the same self-financing holdout accounts used by every investment metric.
- **Settings:** AUM and average exposure use UTC calendar closes; trades are same-direction position episodes, with equal-weighted holding periods. Frequency uses the full account span, including flat periods. Intraday-only strategies can have zero close-sampled leverage despite nonzero intraday exposure; maximum dollar position uses all observed prices.
- **Data:** Use the Bet Sizing portfolio ledger and reconstructed trades. Daily correlation compares account returns with matching AAPL price returns.
- **Decision:** Follow the [portfolio accounting decisions](../../docs/decisions.md#self-financing-portfolio-evaluation).


In [ ]:
general_rows = []
for strategy in strategies:
    ledger = ledgers[strategy]
    days = daily[strategy]
    bets = trades[strategy]
    positions = bets["side"]
    general_rows.append({
        "strategy": strategy,
        "start": start,
        "end": end,
        "average_aum": GeneralCharacteristics.average_aum(days["aum"]),
        "leverage": GeneralCharacteristics.leverage(
            days["position_value"], days["aum"]
        ),
        "maximum_dollar_position_size": max(
            GeneralCharacteristics.maximum_dollar_position_size(
                ledger["position_value_before"]
            ),
            GeneralCharacteristics.maximum_dollar_position_size(
                ledger["position_value"]
            ),
        ),
        "ratio_of_longs": (
            GeneralCharacteristics.ratio_of_longs(positions)
            if len(bets) else np.nan
        ),
        "frequency_of_bets": (
            GeneralCharacteristics.frequency_of_bets(
                positions, bets["event_end"], time_range=(start, end)
            ) if len(bets) else 0.0
        ),
        "average_holding_period": (
            GeneralCharacteristics.average_holding_period(
                positions, bets["event_end"]
            ) if len(bets) else np.nan
        ),
        "annualized_turnover": GeneralCharacteristics.annualized_turnover(
            ledger["traded_value"], days["aum"], time_range=(start, end)
        ),
        "correlation_to_underlying": (
            GeneralCharacteristics.correlation_to_underlying(
                days["net_return"], days["underlying_return"]
            )
        ),
    })
general_characteristics = pd.DataFrame(general_rows).set_index("strategy")
general_characteristics = general_characteristics.reindex(
    columns=registered_metric_order(
        GeneralCharacteristics, general_characteristics.columns
    )
)
display(general_characteristics)


## Performance

- **Purpose:** Report account PnL, annualized return, and closed-trade outcomes.
- **Settings:** Annualization uses actual elapsed calendar time. Each same-direction position episode is one trade.
- **Data:** Include opening, resizing, reversal, and final liquidation costs from the ledger.
- **Decision:** Dollar PnL and trade outcomes replace sums and compounding of overlapping event returns.


In [ ]:
performance_rows = []
for strategy in strategies:
    ledger = ledgers[strategy]
    days = daily[strategy]
    bets = trades[strategy]
    bet_returns = bets["net_return"]
    final_aum = ledger["aum"].iloc[-1]
    net_pnl = Performance.pnl(
        ledger["gross_pnl"] - ledger["execution_cost"]
    )
    np.testing.assert_allclose(
        final_aum - initial_aum[strategy], net_pnl, atol=1e-8
    )
    np.testing.assert_allclose(bets["net_pnl"].sum(), net_pnl, atol=1e-8)
    performance_rows.append({
        "strategy": strategy,
        "pnl": net_pnl,
        "pnl_from_long_positions": (
            Performance.pnl_from_long_positions(
                bets["net_pnl"], bets["side"]
            )
        ),
        "pnl_from_short_positions": (
            Performance.pnl_from_short_positions(
                bets["net_pnl"], bets["side"]
            )
        ),
        "annualized_rate_of_return": (
            Performance.annualized_rate_of_return(
                days["net_return"], start=start, end=end
            )
        ),
        "hit_ratio": (
            Performance.hit_ratio(bet_returns) if len(bets) else np.nan
        ),
        "average_return_from_hits": (
            Performance.average_return_from_hits(bet_returns)
            if len(bets) else np.nan
        ),
        "average_return_from_misses": (
            Performance.average_return_from_misses(bet_returns)
            if len(bets) else np.nan
        ),
    })
performance = pd.DataFrame(performance_rows).set_index("strategy")
performance = performance.reindex(
    columns=registered_metric_order(Performance, performance.columns)
)
display(performance)


## Runs

- **Purpose:** Measure trade-return concentration and full-resolution account drawdowns.
- **Settings:** HHI uses closed-trade returns and trade entry months; drawdown episodes run from the prior peak to recovery or the account end. Percentiles use 95%.
- **Data:** Include initial capital and all pre/post-transaction AUM observations, so entry costs and intraday losses are visible.
- **Decision:** Time under water is measured in years; undefined HHI and episode percentiles remain NaN.


In [ ]:
runs_rows = []
for strategy in strategies:
    bet_returns = trades[strategy]["net_return"]
    aum_path = equity[strategy]
    runs_rows.append({
        "strategy": strategy,
        "hhi_positive_returns": (
            Runs.hhi_positive_returns(bet_returns)
            if len(bet_returns) else np.nan
        ),
        "hhi_negative_returns": (
            Runs.hhi_negative_returns(bet_returns)
            if len(bet_returns) else np.nan
        ),
        "hhi_time_between_bets": (
            Runs.hhi_time_between_bets(bet_returns)
            if len(bet_returns) else np.nan
        ),
        "percentile_drawdown": Runs.percentile_drawdown(aum_path),
        "percentile_time_under_water": (
            Runs.percentile_time_under_water(aum_path)
        ),
    })
runs = pd.DataFrame(runs_rows).set_index("strategy")
runs = runs.reindex(columns=registered_metric_order(Runs, runs.columns))
display(runs)


## Implementation Shortfall

- **Purpose:** Compare execution costs and dollar performance against actual traded notional.
- **Settings:** Transaction notional includes entry, resizing, reversal, and liquidation. Apply 1 bp broker fees and 1 bp slippage one way, symmetrically to buys and sells; no taxes or other costs are added.
- **Data:** Use ledger dollar PnL, traded value, separately recorded broker fees and slippage, and their total execution cost.
- **Decision:** Report all four AFML implementation-shortfall measures; dollar performance is net of total execution costs.


In [ ]:
shortfall_rows = []
for strategy in strategies:
    ledger = ledgers[strategy]
    gross = ledger["gross_pnl"]
    broker_fee = ledger["broker_fee"]
    slippage_cost = ledger["slippage_cost"]
    execution_cost = ledger["execution_cost"]
    net_dollar_performance = gross - execution_cost
    traded_value = ledger["traded_value"]
    shortfall_rows.append({
        "strategy": strategy,
        "broker_fees_per_turnover": (
            ImplementationShortfall.broker_fees_per_turnover(
                broker_fee, traded_value
            )
        ),
        "average_slippage_per_turnover": (
            ImplementationShortfall.average_slippage_per_turnover(
                slippage_cost, traded_value
            )
        ),
        "dollar_performance_per_turnover": (
            ImplementationShortfall.dollar_performance_per_turnover(
                net_dollar_performance, traded_value
            )
        ),
        "return_on_execution_costs": (
            ImplementationShortfall.return_on_execution_costs(
                net_dollar_performance, execution_cost
            )
        ),
    })
implementation_shortfall = pd.DataFrame(shortfall_rows).set_index("strategy")
implementation_shortfall = implementation_shortfall.reindex(
    columns=registered_metric_order(
        ImplementationShortfall, implementation_shortfall.columns
    )
)
display(implementation_shortfall)


## Efficiency

- **Purpose:** Evaluate daily account return variability and the probability that annualized Sharpe exceeds one.
- **Settings:** Sample UTC calendar-day closes including carried-forward non-trading days and first/last partial days. Annualize with 365.25; subtract the effective 3% annual risk-free return over each actual interval from the entire account return.
- **Data:** Account cash earns no interest in the ledger. The first return includes opening costs relative to initial capital.
- **Decision:** Use the portfolio-specific Efficiency interface.


In [ ]:
efficiency = pd.DataFrame.from_dict({
    strategy: Efficiency.portfolio_statistics(
        days["net_return"],
        days["period_start"],
        annual_risk_free_rate=annual_risk_free_rate,
        periods_per_year=365.25,
        annualized_benchmark_sharpe_ratio=annualized_benchmark_sharpe_ratio,
    )
    for strategy, days in daily.items()
}, orient="index")
efficiency = efficiency.rename(
    columns={"annualized_sharpe": "annualized_sharpe_ratio"}
)
efficiency = efficiency.reindex(
    columns=registered_metric_order(Efficiency, efficiency.columns)
)
efficiency.index.name = "strategy"
display(efficiency)


## Classification Scores

- **Purpose:** Evaluate the frozen primary direction classifier and meta action classifier on the holdout labels.
- **Settings:** All scores use the stored event sample weights; positive labels are `1`; negative log loss uses the complete probability vectors for primary labels `[-1, 1]` and meta labels `[0, 1]`.
- **Data:** Use the saved holdout labels, predictions, probability vectors, and sample weights.
- **Decision:** Report accuracy, precision, recall, F1, and negative log loss because the stored artifact directly supports them.

In [ ]:
classification_inputs = {
    "primary": {
        "y_true": holdout["direction_label"],
        "y_pred": holdout["primary_side"],
        "y_pred_proba": pd.DataFrame(
            {
                -1: 1.0 - holdout["primary_probability"],
                1: holdout["primary_probability"],
            },
            index=holdout.index,
        ),
        "labels": [-1, 1],
    },
    "meta": {
        "y_true": holdout["meta_label"],
        "y_pred": holdout["meta_action"],
        "y_pred_proba": pd.DataFrame(
            {
                0: 1.0 - holdout["meta_probability"],
                1: holdout["meta_probability"],
            },
            index=holdout.index,
        ),
        "labels": [0, 1],
    },
}

classification_columns = {}
for model, inputs in classification_inputs.items():
    y_true = inputs["y_true"]
    y_pred = inputs["y_pred"]
    classification_columns[model] = {
        "accuracy": ClassificationScores.accuracy(
            y_true, y_pred, sample_weight=holdout["sample_weight"]
        ),
        "precision": ClassificationScores.precision(
            y_true, y_pred, sample_weight=holdout["sample_weight"]
        ),
        "recall": ClassificationScores.recall(
            y_true, y_pred, sample_weight=holdout["sample_weight"]
        ),
        "f1_score": ClassificationScores.f1_score(
            y_true, y_pred, sample_weight=holdout["sample_weight"]
        ),
        "negative_log_loss": ClassificationScores.negative_log_loss(
            y_true,
            inputs["y_pred_proba"],
            labels=inputs["labels"],
            sample_weight=holdout["sample_weight"],
        ),
    }

classification = pd.DataFrame(classification_columns)
classification = classification.reindex(
    index=registered_metric_order(
        ClassificationScores, classification.index
    )
)
display(classification)
